## Primer on Amazon S3 (Simple Storage Service)
Amazon S3 (Simple Storage Service) is a cloud-based object storage system designed for storing and retrieving any amount of data, at any time, from anywhere. It is hence my central data lake in the ML pipeline covering the following stages: 
- Data ingestion: store raw JSON, CSVs, logs, sensor data etc 
- Data processing: Dump intermediate processed files 
- Model training: Load training datasets, save trained models 
- Model deployment: Serve models, retrieve config, log results 
- Analytics: Store output reports, dashboard, logs 
Note that S3 is scalable, secure, and integrates with all major ML tools (Airflow, Spark, PyTorch, HuggingFace etc). 

To access S3 using Python, we need to install and import boto3. This is the official Python SDK (software development kit) for Amazon Web Services (AWS). It lets Python developers programmatically interact with AWS services like 
- S3 for storing/retrieving data and models 
- EC2 for launching cloud servers
- Lambda for running serverless functions 
- SageMaker for training and deploying ML models 

There are two main ways we interact with AWS services using boto3: 
- Client: for low-level direct API calls (e.g. boto3.client('s3'))
    - this usually returns raw dictionaries and is more configurable in terms of control and flexibility, hence preferred use 
- Resource: higher-level, Pythonic objects (e.g. boto3.resource('s3'))
    - this usually returns objects like Bucket, Object, which are rich Python objects with attributes/methods but less granular control

Hence, in a full-stack ML workflow, boto3 allows you to: 
- Automate uploads (e.g. saving processed CSVs, model files to S3)
- Pull training data (e.g. read large datasets from S3 directly into pandas or PyTorch)
- Trigger jobs (e.g. launch an EC2 instance or SageMaker training job)
- Manage infroa (e.g. spin up or tear down cloud resources for experiments)
- Secure access (e.g. use IAM credentials to enforce permissions programmatically)

### Step 0: Set-up and Authentication 
1. in CLI, run pip install boto3 pandas s3fs
2. Then, three options to configure AWS credentials
    1. CLI-based (recommended): aws configure 
        - This will trigger the AWS CLI to ask for 
            - AWS Access Key ID [None]: <your_access_key>
            - AWS Secret Access Key [None]: <your_secret_key>
            - Default region name [None]: ap-southeast-1
            - Default output format [None]: json
        - Then it will store my credentials in ~/.aws/credentials (first two) and ~/.aws/config (last two)
        - Then in my code, I can skip all auth set-up as per point 3, since boto3 automatically looks for auth credentials in those files
    2. In-code set-up below
    3. Environment variables by running in CLI 
        - export AWS_ACCESS_KEY_ID=your_key
        - export AWS_SECRET_ACCESS_KEY=your_secret    
3. boto3 automatically looks for credentials in a specific order 
    - Environment variables (e.g. AWS_ACCESS_KEY_ID)
    - AWS config file (~/.aws/credentials) via aws configure
    - IAM roles (when running in EC2, SageMaker)
4. Hence once authenticated, can safely and easily access cloud resources in Python code, both locally and in production
5. Note that the AWS Credentials are generated from my AWS account. There are two options 
    - Option A: Root User (not recommended)
        - Log-in to aws.amazon.com
        - Go to My Security Credentials 
        - Generate Access Key ID + Secret Access Key 
    - Option B: IAM User (recommended - especially if part of team/org)
        - Admin creates an IAM (Identity and Access Management) user for me 
        - Log into IAM Console
        - Under Security credentials, click create access key, which will give 
            - AWS_ACCESS_KEY_ID: public identifier
            - AWS_SECRET_ACCESS_KEY: secret string (download once)
        - Note: creating IAM users preferred because this only gives people the permission they need (principle of least privilege)
            - Similar to master access card and specific room key cards 
6. Also note that I will need to install the AWS CLI, which is a tool for me to control and manage AWS services from my terminal instead of using the AWS web console
    - To install, for macOS use brew install awscli
        - Recall that Homebrew (as called by brew) is a package manager for macOS that lets you install, update, and manage software tools from the command line. This is similar to pip for Python but for your entire system
            - System-wide tools are those available across entire computer, written in any language and used via terminal commands, vs Python tools that are imported in Python scrips and installed for a specific Python interpreter or virtual environment
            - For pip freeze > requirements.txt, this only captures Python packages installed via pip in current Python environment. Does not cover system tools installed via brew, conda-native packages if using conda install, or OS-level dependencies
                - for conda, do conda env export > environment.yml 
                - for brew, do brew bundle dump --file=Brewfile but note that it is less common to share
        - Key commands are 
            - brew search awscli (search for package)
            - brew install awscli (install package)
            - brew upgrade awscli (upgrade package)
            - brew uninstall awscli (uninstall package)
            - brew list (list everything you've installed)
    - Main aws cli commands are 
        - aws configure #set-up credentials 
        - aws s3 ls #list buckets
        - aws s3 ls s3://my-bucket/data/ #list files
        - aws s3 cp file.csv s3://my-bucket/ # Upload

In [2]:
import boto3

#Option 2: Hardcode in Code for manual client setup. Whilst this works even without AWS CLI, it is not secure and not scalable for shared use
s3 = boto3.client(
    's3', 
    aws_access_key_id='Your_Access_key', 
    aws_secret_access_key='Your_Secret_key', 
    region_name='ap-southeast-1'
)

#### Common S3 Operations with boto3
Terminology recap 
- Bucket: folder or drive which is the top-level container in S3 (e.g. my-ml-project)
- Key: full file path inside a bucket (e.g. data/raw/customers_2024.csv)
- Object: actual file / data blob stored at that key (e.g. customers_2024.csv)
- Region: AWS datacenter zone corresponding to the physical AWS data center where is bucket is hosted (e.g. ap-southeast-1)

This corresponds to the following full S3 URL: s3://my-ml-project/data/raw/customers_2024.csv

In [ ]:
#Upload file to S3: s3.upload_file(Filename, Bucket, Key)
s3.upload_file('local.csv', 'my-bucket', 'data/raw/local.csv')

#Download file from S3: s3.download_file(Bucket, Key, Filename)
s3.download_file('my-bucket', 'data/raw/local.csv', 'local_copy.csv')

In [ ]:
#List files in a folder (client)
response = s3.list_objects_v2(Bucket='my_bucket', Prefix='data/') #give me everything inside my_bucket that starts with data/ (latter acts like a folder path filter). This returns a response like { 'Contents':[{'Key': 'data/file1.csv'}, {'Key': 'data/file2.csv'}]}
for obj in response.get('Contents', []): #response.get("Contents, []") extracts list of objects from the response. if use .get() method for dictionaries to retrieve the value associated for the key, even if Contents is missing (no files), won't crash but return [] instead. syntax is response.get(key, default_value). This (dict.get()) is very different from requests.get(url) which is to send a get request to a URL e.g. in an API call
    print(obj['Key']) #loop thru each object (file metadata) and print the full key path

#List files (resource)
bucket = boto3.resource('s3').Bucket('my-bucket')
for obj in bucket.objects.filter(Prefix='data/'): 
    print(obj.key)

In [ ]:
#Read CSV from S3 into pandas 
import pandas as pd 
df = pd.read_csv('s3://my-bucket/data/raw/local.csv') #have to insert the full s3 URL